In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [32]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler,LabelEncoder
df = pd.read_csv('/kaggle/input/datasets/ayesha73/datefruit/DateFruit_Dataset.csv')
df.head()
X=df.drop("Class",axis=1)
y=df["Class"]
le=LabelEncoder()
y=le.fit_transform(y)

In [33]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [34]:
# Defining the ANN Model
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset

In [38]:
X_train_tesnor=torch.tensor(X_train_scaled,dtype=torch.float32)
X_test_tensor=torch.tensor(X_test_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train,dtype=torch.long)
y_test_tensor=torch.tensor(y_test,dtype=torch.long)
train_dataset=TensorDataset(X_train_tesnor,y_train_tesnor)
test_dataset=TensorDataset(X_test_tensor,y_test_tensor)
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32)

In [40]:
# Defining the Architecture of the ANN Model
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()
        self.model=nn.Sequential(
            # 1st Hidden Layer
            nn.Linear(X.shape[1],64),
            nn.ReLU(),
            # 2nd Hidden Layer
            nn.Linear(64,64),
            nn.ReLU(),
            # Outpur Layer
            nn.Linear(64,7)   
        )
    def forward(self,x):
            return self.model(x)


model=ANN()
criteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [41]:
# Training the Model
epochs=100
for epoch in range(epochs):
    model.train()
    running_loss=0.0
    for xb,yb in train_loader:
        optimizer.zero_grad()
        outputs=model(xb)
        loss=criteria(outputs,yb)
        loss.backward()
        optimizer.step() #params update
        running_loss+=loss.item()
    train_loss=running_loss/len(train_loader)
    print(f"epoch={epoch+1}/{epochs} ,loss={train_loss}")

epoch=1/100 ,loss=1.7422655095224795
epoch=2/100 ,loss=1.136429110298986
epoch=3/100 ,loss=0.7296128519203352
epoch=4/100 ,loss=0.5496705666832302
epoch=5/100 ,loss=0.4549071037251016
epoch=6/100 ,loss=0.3987500550954238
epoch=7/100 ,loss=0.3455829037272412
epoch=8/100 ,loss=0.31632363472295844
epoch=9/100 ,loss=0.29448939989442413
epoch=10/100 ,loss=0.27904545677744824
epoch=11/100 ,loss=0.2538594188897506
epoch=12/100 ,loss=0.23043063444935757
epoch=13/100 ,loss=0.2159041214896285
epoch=14/100 ,loss=0.2074277214705944
epoch=15/100 ,loss=0.19492738985497018
epoch=16/100 ,loss=0.19268616515657175
epoch=17/100 ,loss=0.17864406821520432
epoch=18/100 ,loss=0.17037370412246042
epoch=19/100 ,loss=0.16400491430059724
epoch=20/100 ,loss=0.1587323397397995
epoch=21/100 ,loss=0.14969868650254997
epoch=22/100 ,loss=0.15429306240833324
epoch=23/100 ,loss=0.14362596963410793
epoch=24/100 ,loss=0.1598417562958987
epoch=25/100 ,loss=0.14323342332373495
epoch=26/100 ,loss=0.13064654722161914
epoch=27

In [42]:
# Evaluating the Model
model.eval()
total=0
correct=0
with torch.no_grad():
    for xb,yb in test_loader:
        outputs=model(xb) ## [0.2, 0.5, 1.3, -0.5, ..] - 7 vals
        _,predicted=torch.max(outputs,1)
        correct+= (predicted==yb).sum().item()
        total+=yb.size(0)
print("Accuracy:",correct/total * 100)

Accuracy: 95.55555555555556
